# Content That Converts - Advanced ML & Evaluation (V2)

This fully updated notebook demonstrates our **Enterprise-Grade** approach to analyzing Groupon deals, shifting from naive statistical correlations to **Random Forest Feature Importances**, **Multi-Agent Evaluation Rubrics**, and **Chain-of-Thought Asynchronous Generation**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
from sklearn.ensemble import RandomForestRegressor
import textstat
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", context="talk")


## 1. Advanced NLP Feature Extraction
We parse `deals.csv`, overlaying deep NLP metrics such as Flesch-Kincaid Readability and Sentiment.


In [ ]:
df = pd.read_csv('../data/deals.csv')
analyzer = SentimentIntensityAnalyzer()
    
# Basic metrics
df['title_length'] = df['title'].fillna('').apply(len)
df['desc_word_count'] = df['description'].fillna('').apply(lambda x: len(str(x).split()))
df['fine_print_length'] = df['fine_print'].fillna('').apply(len)

# NLP metrics
df['desc_sentiment'] = df['description'].fillna('').apply(lambda x: analyzer.polarity_scores(str(x))['compound'])
df['desc_readability'] = df['description'].fillna('').apply(lambda x: textstat.flesch_reading_ease(str(x)))

# Urgency / Value signals
urgency_words = ['now', 'today', 'hurry', 'limited', 'quickly', 'opportunity']
value_words = ['save', 'value', 'discount', 'fraction', 'affordable']
df['urgency_score'] = df['description'].fillna('').str.lower().apply(lambda x: sum(1 for w in urgency_words if w in x))
df['value_score'] = df['description'].fillna('').str.lower().apply(lambda x: sum(1 for w in value_words if w in x))

features = ['title_length', 'desc_word_count', 'fine_print_length', 'image_quality_score', 
            'price', 'discount_pct', 'desc_sentiment', 'desc_readability', 'urgency_score', 'value_score']

display(df[['deal_id', 'cvr'] + features].head(3))


## 2. Machine Learning: Random Forest Importances
Rather than relying on basic Pearson Correlation, we use an RF to learn the non-linear boundaries.


In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=42, max_depth=5)
X = df[features].fillna(0)
y = df['cvr'].fillna(df['cvr'].mean())
rf.fit(X, y)

feature_importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances.values, y=feature_importances.index, palette="viridis")
plt.title("Random Forest Modeling: What Drives Conversion?")
plt.xlabel("Feature Importance (Gini)")
plt.show()


## 3. Asynchronous Model Evaluation
Our LLM generated new content asynchronously using **Pydantic** constraints and **Instructor**. Our multi-metric automated evaluator scored the results based on Persuasiveness, Clarity, and Hallucination Risk.


In [ ]:
with open('../results/poc_results.json', 'r') as f:
    poc_results = json.load(f)

eval_data = []
for r in poc_results:
    metrics = r['eval_metrics']
    multi = metrics.get('multi_agent_eval', {})
    eval_data.append({
        'deal_id': r['original']['deal_id'][:8],
        'Persuasiveness': multi.get('persuasiveness_score', 0),
        'Clarity': multi.get('clarity_score', 0),
        'Hallucination_Penalty': multi.get('hallucination_penalty', 0),
        'Winner': multi.get('judge_verdict', 'UNKNOWN')
    })
    
eval_df = pd.DataFrame(eval_data)
display(eval_df)


In [ ]:
# Visualize Evaluation Metrics
eval_melted = eval_df.melt(id_vars='deal_id', value_vars=['Persuasiveness', 'Clarity'], var_name='Metric', value_name='Score (out of 10)')

plt.figure(figsize=(10, 5))
sns.barplot(data=eval_melted, x='deal_id', y='Score (out of 10)', hue='Metric', palette='magma')
plt.title("LLM Deal Evaluation: Persuasiveness & Clarity Scores")
plt.ylim(0, 10.5)
plt.axhline(8, ls='--', color='green', label='Success Threshold')
plt.legend(loc='lower right')
plt.show()
